## Single Cell Example

This example downloads the `pancreas` differentiation dataset and fits an NB model like we've shared before. You can filter the number of genes using `pancreas = scdesigner.dataset.pancreas()[:, :n_genes]` but even with 1000 genes the computation is not too bad.

In [ ]:
from scdesigner.simulators import NegBinCopula
import scdesigner.datasets

pancreas = scdesigner.datasets.pancreas()
pancreas

This fits the model.

In [ ]:
sim = NegBinCopula(mean_formula="bs(pseudotime, df=10)", dispersion_formula="bs(pseudotime, df=4)")
sim.fit(pancreas, max_epochs=50)

The sampled output is another anndata object.

In [ ]:
samples = sim.sample()
samples

### Visual Checks [Optional]

We have a few helper functions to check the simulator, which we've placed in the `scdiagnostics` package. In the website, this will be replaced by more advanced interactive versions. `scdiagnostics` depends on the `altair` package, which might make installation difficult. The remaining dataset examples all work without the visualization steps, which is why we've indicated this section as "optional."

In [ ]:
import scdiagnostics as scd

scd.compare_umap(pancreas, samples, color="cell_type")

In [ ]:
_  = scd.compare_boxplot(pancreas, samples, max_plot=20)

## Spatial Data Example

This is a spatial transcriptomics example using the acinar cell carcinoma dataset from the scDesign3 zenodo repository.

In [ ]:
from scdesigner.simulators import SpatialNegBinCopula
import scdesigner.datasets

acinar = scdesigner.datasets.acinar()
acinar

This is our NB copula that automatically derives a spatial basis. We've set `basis="tps"` in this example because the default `"gp"` is too smooth for this dataset.

In [ ]:
sim = SpatialNegBinCopula(mean_df=400, basis="tps", mean_extra_terms="cell_type")
sim.fit(acinar, max_epochs=50)

As before, the simulated output is an anndata object.

In [ ]:
samples = sim.sample()
samples

### Visual Checks [Optional]

We'll make the same visualizations as above, but with an extra to compare the spatial expression patterns.

In [ ]:
scd.compare_umap(acinar, samples, color="cell_type")

In [ ]:
_ = scd.compare_boxplot(acinar, samples, max_plot=20)

In [ ]:
[display(scd.plot_mean_surface(sim, acinar, gene=g)) for g in acinar.var_names[:3]]

## scATAC-seq Example

This is the single-cell ATAC-seq dataset from the signac package, also from the scDesign3 zenodo reposiitory. Like in the scDesign3 paper, we use ZINB marginals. Note that the actual data have been preprocessed, though, and aren't exactly integer valued. The ZINB gives a reasonable approximation, but will always be constrained to integers.

In [ ]:
from scdesigner.simulators import ZeroInflatedNegBinCopula

granja = scdesigner.datasets.granja_atac()
granja

We have to specify formulas for each of the components of the ZINB model. The default learning rate is also a little too high for this example.

In [ ]:
sim = ZeroInflatedNegBinCopula(mean_formula="~ cell_type", dispersion_formula="~ cell_type", zero_inflation_formula="~ cell_type")
sim.fit(granja, max_epochs=60, lr=0.1)

In [ ]:
samples = sim.sample()
samples

### Visual Checks [Optional]

In [ ]:
scd.compare_umap(granja, samples, color="cell_type")

In [ ]:
_ = scd.compare_boxplot(granja, samples, max_plot=20)